#Data Aggregation and Group Operations
After loading, merging, and preparing a dataset

pandas provides a versatile groupby interface, enabling you to slice, dice, and summarize datasets in a natural way

In [ ]:
import pandas as pd
import numpy as np

# Group Operations
**split-apply-combine**

In [ ]:
df = pd.DataFrame({"key1" : ["a", "a", None, "b", "b", "a", None],
                   "key2" : pd.Series([1, 2, 1, 2, 1, None, 1], dtype="Int64"),
                   "data1" : np.random.standard_normal(7),
                   "data2" : np.random.standard_normal(7)})
df

,key1,key2,data1,data2
0,a,1,-0.428484,0.429911
1,a,2,-0.300281,-0.346383
2,None,1,0.507235,-1.029824
3,b,2,-0.230104,0.698853
4,b,1,-0.976782,0.377634
5,a,<NA>,-0.914970,1.512513
6,None,1,-0.227159,-0.327490


##.groupby()
The idea is that this object has all of the information needed to then apply some operation to each of the groups.

In [ ]:
grouped = df["data1"].groupby(df["key1"])
grouped

##.mean()

In [ ]:
grouped.mean()

,data1
key1,
a,-0.547912
b,-0.603443


In [ ]:
means = df["data1"].groupby([df["key1"], df["key2"]]).mean()
means

key1  key2
a     1      -0.428484
      2      -0.300281
b     1      -0.976782
      2      -0.230104
Name: data1, dtype: float64

In [ ]:
means.unstack()

key2,1,2
key1,,
a,-0.428484,-0.300281
b,-0.976782,-0.230104


In [ ]:
states = np.array(["OH", "CA", "CA", "OH", "OH", "CA", "OH"])
years = [2005, 2005, 2006, 2005, 2006, 2005, 2006]
df["data1"].groupby([states, years]).mean()

CA  2005   -0.607625
    2006    0.507235
OH  2005   -0.329294
    2006   -0.601970
Name: data1, dtype: float64

In [ ]:
df.groupby("key1").mean()

,key2,data1,data2
key1,,,
a,1.5,-0.547912,0.532014
b,1.5,-0.603443,0.538244


In [ ]:
df.groupby("key2").mean(numeric_only=True)

,data1,data2
key2,,
1,-0.281297,-0.137442
2,-0.265192,0.176235


.mean(numerical_only=True)

In [ ]:
df.groupby(["key1", "key2"]).mean()

data1     data2
key1 key2                    
a    1    -0.428484  0.429911
     2    -0.300281 -0.346383
b    1    -0.976782  0.377634
     2    -0.230104  0.698853

##.size()

In [ ]:
df.groupby(["key1", "key2"]).size()

key1  key2
a     1       1
      2       1
b     1       1
      2       1
dtype: int64

In [ ]:
df.groupby("key1", dropna=True ).size()

,0
key1,
a,3
b,2


In [ ]:
df.groupby("key1", dropna=False ).size()

,0
key1,
a,3
b,2
NaN,2


In [ ]:
df.groupby(["key1", "key2"], dropna=False).size()

key1  key2
a     1       1
      2       1
      <NA>    1
b     1       1
      2       1
NaN   1       2
dtype: int64

##.count()

In [ ]:
df.groupby("key1").count()

,key2,data1,data2
key1,,,
a,2,3,3
b,2,2,2


## Iterating over Groups
**groupby supports iteration**

In [ ]:
for name, group in df.groupby("key1"):
  print(name)
  print(group)

a
  key1  key2     data1     data2
0    a     1 -0.428484  0.429911
1    a     2 -0.300281 -0.346383
5    a  <NA> -0.914970  1.512513
b
  key1  key2     data1     data2
3    b     2 -0.230104  0.698853
4    b     1 -0.976782  0.377634


In [ ]:
for (k1, k2), group in df.groupby(["key1", "key2"]):
  print((k1, k2))
  print(group)

('a', np.int64(1))
  key1  key2     data1     data2
0    a     1 -0.428484  0.429911
('a', np.int64(2))
  key1  key2     data1     data2
1    a     2 -0.300281 -0.346383
('b', np.int64(1))
  key1  key2     data1     data2
4    b     1 -0.976782  0.377634
('b', np.int64(2))
  key1  key2     data1     data2
3    b     2 -0.230104  0.698853


In [ ]:
pieces = {name: group for name, group in df.groupby("key1")}
pieces["b"]

,key1,key2,data1,data2
3,b,2,-0.230104,0.698853
4,b,1,-0.976782,0.377634


In [ ]:
grouped = df.groupby({"key1": "key", "key2": "key",
                      "data1": "data", "data2": "data"}, axis="columns")
grouped

<ipython-input-19-101400185bc6>:1: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  grouped = df.groupby({"key1": "key", "key2": "key",


In [ ]:
for group_key, group_values in grouped:
  print(group_key)
  print(group_values)

data
      data1     data2
0 -0.428484  0.429911
1 -0.300281 -0.346383
2  0.507235 -1.029824
3 -0.230104  0.698853
4 -0.976782  0.377634
5 -0.914970  1.512513
6 -0.227159 -0.327490
key
   key1  key2
0     a     1
1     a     2
2  None     1
3     b     2
4     b     1
5     a  <NA>
6  None     1


###Selecting a Column or Subset of Columns
Indexing a GroupBy object created from a DataFrame with a column name or array of column names has the effect of column subsetting for aggregation.

In [ ]:
df.groupby("key1")["data1"]
df.groupby("key1")[["data2"]]

In [ ]:
df["data1"].groupby(df["key1"])
df[["data2"]].groupby(df["key1"])

In [ ]:
df.groupby(["key1", "key2"])[["data2"]].mean()

data2
key1 key2          
a    1     0.429911
     2    -0.346383
b    1     0.377634
     2     0.698853

In [ ]:
s_grouped = df.groupby(["key1", "key2"])["data2"]
s_grouped

In [ ]:
s_grouped.mean()

key1  key2
a     1       0.429911
      2      -0.346383
b     1       0.377634
      2       0.698853
Name: data2, dtype: float64

##Grouping with Dictionaries and Series

In [ ]:
people = pd.DataFrame(np.random.standard_normal((5, 5)),
                      columns=["a", "b", "c", "d", "e"],
                      index=["Joe", "Steve", "Wanda", "Jill", "Trey"])
people.iloc[2:3, [1, 2]] = np.nan # Add a few NA values
people

,a,b,c,d,e
Joe,0.377964,-1.051753,0.762790,-1.085117,1.310027
Steve,0.236744,-0.976804,0.590052,-0.118339,0.453154
Wanda,0.694452,NaN,NaN,0.877755,-0.127143
Jill,0.949145,-1.618264,-0.021685,-0.530906,1.984372
Trey,-2.508122,-0.024784,-0.342975,0.170406,1.016886


In [ ]:
mapping = {"a": "red", "b": "red", "c": "blue",
           "d": "blue", "e": "red", "f" : "orange"}
by_column = people.groupby(mapping, axis="columns")
by_column.sum()

<ipython-input-27-3a9a9b2c49cc>:3: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  by_column = people.groupby(mapping, axis="columns")


,blue,red
Joe,-0.322327,0.636239
Steve,0.471713,-0.286906
Wanda,0.877755,0.567309
Jill,-0.552591,1.315252
Trey,-0.172570,-1.516020


In [ ]:
map_series = pd.Series(mapping)
map_series

,0
a,red
b,red
c,blue
d,blue
e,red
f,orange


In [ ]:
people.groupby(map_series, axis="columns").count()

<ipython-input-29-311085072b25>:1: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  people.groupby(map_series, axis="columns").count()


,blue,red
Joe,2,3
Steve,2,3
Wanda,1,2
Jill,2,3
Trey,2,3


##Grouping with Functions
**len function**

In [ ]:
people.groupby(len).sum()
#take as index the name as the first column

,a,b,c,d,e
3,0.377964,-1.051753,0.762790,-1.085117,1.310027
4,-1.558977,-1.643048,-0.364660,-0.360500,3.001257
5,0.931195,-0.976804,0.590052,0.759416,0.326011


In [ ]:
key_list = ["one", "one", "one", "two", "two"]
people.groupby([len, key_list]).min()

,,a,b,c,d,e
3,one,0.377964,-1.051753,0.762790,-1.085117,1.310027
4,two,-2.508122,-1.618264,-0.342975,-0.530906,1.016886
5,one,0.236744,-0.976804,0.590052,-0.118339,-0.127143


##Grouping by Index Levels

In [ ]:
columns = pd.MultiIndex.from_arrays([["US", "US", "US", "JP", "JP"],
                                      [1, 3, 5, 1, 3]],
                                    names=["cty", "tenor"])
hier_df = pd.DataFrame(np.random.standard_normal((4, 5)), columns=columns)
hier_df

cty          US                            JP          
tenor         1         3         5         1         3
0     -0.160012 -0.736892 -0.284289  0.300752  0.516154
1      0.683637  1.003565 -0.902553  0.168375  0.257255
2      0.662664  0.792166  0.002945  0.112494  2.349158
3     -1.070335  0.020691  0.224218 -0.946910 -0.107210

In [ ]:
key_list = ["one", "one", "one", "two", "two"]
people.groupby([len, key_list]).min()

,,a,b,c,d,e
3,one,0.377964,-1.051753,0.762790,-1.085117,1.310027
4,two,-2.508122,-1.618264,-0.342975,-0.530906,1.016886
5,one,0.236744,-0.976804,0.590052,-0.118339,-0.127143


#Data Aggregation
**Aggregations** refer to any data transformation that produces scalar values from arrays
mean, count, min, and sum

In [ ]:
df

,key1,key2,data1,data2
0,a,1,-0.428484,0.429911
1,a,2,-0.300281,-0.346383
2,None,1,0.507235,-1.029824
3,b,2,-0.230104,0.698853
4,b,1,-0.976782,0.377634
5,a,<NA>,-0.914970,1.512513
6,None,1,-0.227159,-0.327490


In [ ]:
grouped = df.groupby("key1")
grouped["data1"].nsmallest(2)

key1   
a     5   -0.914970
      0   -0.428484
b     4   -0.976782
      3   -0.230104
Name: data1, dtype: float64

In [ ]:
grouped = df.groupby("key1")
grouped["data1"]

##.agg()

In [ ]:
def peak_to_peak(arr):
  return arr.max() - arr.min()
grouped.agg(peak_to_peak)

,key2,data1,data2
key1,,,
a,1,0.614690,1.858895
b,1,0.746678,0.321220


In [ ]:
grouped.describe()

count      mean       std       min       25%       50%       75%  \
data data1   7.0 -0.367221  0.497145 -0.976782 -0.671727 -0.300281 -0.228631   
     data2   7.0  0.187888  0.831081 -1.029824 -0.336936  0.377634  0.564382   
key  key2    6.0  1.333333  0.516398       1.0       1.0       1.0      1.75   

                 max  
data data1  0.507235  
     data2  1.512513  
key  key2        2.0

##Column-Wise and Multiple Function Application

In [52]:
tips = pd.read_csv("examples/tips.csv")
tips.head()

,total_bill,tip,smoker,day,time,size
0,16.99,1.01,No,Sun,Dinner,2
1,10.34,1.66,No,Sun,Dinner,3
2,21.01,3.50,No,Sun,Dinner,3
3,23.68,3.31,No,Sun,Dinner,2
4,24.59,3.61,No,Sun,Dinner,4


In [53]:
tips["tip_pct"] = tips["tip"] / tips["total_bill"]
tips.head()

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808


In [55]:
grouped = tips.groupby(["day", "smoker"])
grouped_pct = grouped["tip_pct"]
grouped_pct.agg("mean")

day   smoker
Fri   No        0.151650
      Yes       0.174783
Sat   No        0.158048
      Yes       0.147906
Sun   No        0.160113
      Yes       0.187250
Thur  No        0.160298
      Yes       0.163863
Name: tip_pct, dtype: float64

If you pass a list of functions or function names instead, you get back a DataFrame with column names taken from the functions

In [56]:
grouped_pct.agg(["mean", "std", peak_to_peak])

mean       std  peak_to_peak
day  smoker                                  
Fri  No      0.151650  0.028123      0.067349
     Yes     0.174783  0.051293      0.159925
Sat  No      0.158048  0.039767      0.235193
     Yes     0.147906  0.061375      0.290095
Sun  No      0.160113  0.042347      0.193226
     Yes     0.187250  0.154134      0.644685
Thur No      0.160298  0.038774      0.193350
     Yes     0.163863  0.039389      0.151240

In [57]:
grouped_pct.agg([("average", "mean"), ("stdev", np.std)])
#You could pass a list of (name, function) tuples

<ipython-input-57-006287cd2d77>:1: FutureWarning: The provided callable <function std at 0x788207506160> is currently using SeriesGroupBy.std. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "std" instead.
  grouped_pct.agg([("average", "mean"), ("stdev", np.std)])


average     stdev
day  smoker                    
Fri  No      0.151650  0.028123
     Yes     0.174783  0.051293
Sat  No      0.158048  0.039767
     Yes     0.147906  0.061375
Sun  No      0.160113  0.042347
     Yes     0.187250  0.154134
Thur No      0.160298  0.038774
     Yes     0.163863  0.039389

In [58]:
functions = ["count", "mean", "max"]
result = grouped[["tip_pct", "total_bill"]].agg(functions)
result

tip_pct                     total_bill                  
              count      mean       max      count       mean    max
day  smoker                                                         
Fri  No           4  0.151650  0.187735          4  18.420000  22.75
     Yes         15  0.174783  0.263480         15  16.813333  40.17
Sat  No          45  0.158048  0.291990         45  19.661778  48.33
     Yes         42  0.147906  0.325733         42  21.276667  50.81
Sun  No          57  0.160113  0.252672         57  20.506667  48.17
     Yes         19  0.187250  0.710345         19  24.120000  45.35
Thur No          45  0.160298  0.266312         45  17.113111  41.19
     Yes         17  0.163863  0.241255         17  19.190588  43.11

In [59]:
result["tip_pct"]

count      mean       max
day  smoker                           
Fri  No          4  0.151650  0.187735
     Yes        15  0.174783  0.263480
Sat  No         45  0.158048  0.291990
     Yes        42  0.147906  0.325733
Sun  No         57  0.160113  0.252672
     Yes        19  0.187250  0.710345
Thur No         45  0.160298  0.266312
     Yes        17  0.163863  0.241255

In [60]:
ftuples = [("Average", "mean"), ("Variance", np.var)]
grouped[["tip_pct", "total_bill"]].agg(ftuples)

<ipython-input-60-c345fa286676>:2: FutureWarning: The provided callable <function var at 0x7882075062a0> is currently using SeriesGroupBy.var. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "var" instead.
  grouped[["tip_pct", "total_bill"]].agg(ftuples)


tip_pct           total_bill            
              Average  Variance    Average    Variance
day  smoker                                           
Fri  No      0.151650  0.000791  18.420000   25.596333
     Yes     0.174783  0.002631  16.813333   82.562438
Sat  No      0.158048  0.001581  19.661778   79.908965
     Yes     0.147906  0.003767  21.276667  101.387535
Sun  No      0.160113  0.001793  20.506667   66.099980
     Yes     0.187250  0.023757  24.120000  109.046044
Thur No      0.160298  0.001503  17.113111   59.625081
     Yes     0.163863  0.001551  19.190588   69.808518

In [61]:
grouped.agg({"tip" : np.max, "size" : "sum"})

<ipython-input-61-4dfd6ef53c71>:1: FutureWarning: The provided callable <function max at 0x788207505620> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped.agg({"tip" : np.max, "size" : "sum"})


tip  size
day  smoker             
Fri  No       3.50     9
     Yes      4.73    31
Sat  No       9.00   115
     Yes     10.00   104
Sun  No       6.00   167
     Yes      6.50    49
Thur No       6.70   112
     Yes      5.00    40

In [62]:
grouped.agg({"tip_pct" : ["min", "max", "mean", "std"],"size" : "sum"})

tip_pct                               size
                  min       max      mean       std  sum
day  smoker                                             
Fri  No      0.120385  0.187735  0.151650  0.028123    9
     Yes     0.103555  0.263480  0.174783  0.051293   31
Sat  No      0.056797  0.291990  0.158048  0.039767  115
     Yes     0.035638  0.325733  0.147906  0.061375  104
Sun  No      0.059447  0.252672  0.160113  0.042347  167
     Yes     0.065660  0.710345  0.187250  0.154134   49
Thur No      0.072961  0.266312  0.160298  0.038774  112
     Yes     0.090014  0.241255  0.163863  0.039389   40

##Returning Aggregated Data Without Row Indexes

In [63]:
grouped = tips.groupby(["day", "smoker"], as_index=False)
grouped.mean(numeric_only=True)

,day,smoker,total_bill,tip,size,tip_pct
0,Fri,No,18.420000,2.812500,2.250000,0.151650
1,Fri,Yes,16.813333,2.714000,2.066667,0.174783
2,Sat,No,19.661778,3.102889,2.555556,0.158048
3,Sat,Yes,21.276667,2.875476,2.476190,0.147906
4,Sun,No,20.506667,3.167895,2.929825,0.160113
5,Sun,Yes,24.120000,3.516842,2.578947,0.187250
6,Thur,No,17.113111,2.673778,2.488889,0.160298
7,Thur,Yes,19.190588,3.030000,2.352941,0.163863


In [64]:
grouped = tips.groupby(["day", "smoker"])
grouped.mean(numeric_only=True)

total_bill       tip      size   tip_pct
day  smoker                                          
Fri  No       18.420000  2.812500  2.250000  0.151650
     Yes      16.813333  2.714000  2.066667  0.174783
Sat  No       19.661778  3.102889  2.555556  0.158048
     Yes      21.276667  2.875476  2.476190  0.147906
Sun  No       20.506667  3.167895  2.929825  0.160113
     Yes      24.120000  3.516842  2.578947  0.187250
Thur No       17.113111  2.673778  2.488889  0.160298
     Yes      19.190588  3.030000  2.352941  0.163863

##Apply:General split-apply-combine


In [72]:
def top(df, n=5, column="tip_pct"):
  return df.sort_values(column, ascending=False)[:n]
top(tips, n=6)

,total_bill,tip,smoker,day,time,size,tip_pct
172,7.25,5.15,Yes,Sun,Dinner,2,0.710345
178,9.60,4.00,Yes,Sun,Dinner,2,0.416667
67,3.07,1.00,Yes,Sat,Dinner,1,0.325733
232,11.61,3.39,No,Sat,Dinner,2,0.291990
183,23.17,6.50,Yes,Sun,Dinner,4,0.280535
109,14.31,4.00,Yes,Sat,Dinner,2,0.279525


In [85]:
a=df.sort_values(column="tip_pct",ascending=False)
a.head()

TypeError: DataFrame.sort_values() got an unexpected keyword argument 'column'

In [70]:
top(tips, n=3)

,total_bill,tip,smoker,day,time,size,tip_pct
172,7.25,5.15,Yes,Sun,Dinner,2,0.710345
178,9.60,4.00,Yes,Sun,Dinner,2,0.416667
67,3.07,1.00,Yes,Sat,Dinner,1,0.325733


##.apply()

In [86]:
tips.groupby("smoker").apply(top)

<ipython-input-86-2a8c000674ed>:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tips.groupby("smoker").apply(top)


total_bill   tip smoker   day    time  size   tip_pct
smoker                                                           
No     232       11.61  3.39     No   Sat  Dinner     2  0.291990
       149        7.51  2.00     No  Thur   Lunch     2  0.266312
       51        10.29  2.60     No   Sun  Dinner     2  0.252672
       185       20.69  5.00     No   Sun  Dinner     5  0.241663
       88        24.71  5.85     No  Thur   Lunch     2  0.236746
Yes    172        7.25  5.15    Yes   Sun  Dinner     2  0.710345
       178        9.60  4.00    Yes   Sun  Dinner     2  0.416667
       67         3.07  1.00    Yes   Sat  Dinner     1  0.325733
       183       23.17  6.50    Yes   Sun  Dinner     4  0.280535
       109       14.31  4.00    Yes   Sat  Dinner     2  0.279525

In [87]:
result = tips.groupby("smoker")["tip_pct"].describe()
result

,count,mean,std,min,25%,50%,75%,max
smoker,,,,,,,,
No,151.0,0.159328,0.039910,0.056797,0.136906,0.155625,0.185014,0.291990
Yes,93.0,0.163196,0.085119,0.035638,0.106771,0.153846,0.195059,0.710345


In [88]:
result.unstack("smoker")

smoker
count  No        151.000000
       Yes        93.000000
mean   No          0.159328
       Yes         0.163196
std    No          0.039910
       Yes         0.085119
min    No          0.056797
       Yes         0.035638
25%    No          0.136906
       Yes         0.106771
50%    No          0.155625
       Yes         0.153846
75%    No          0.185014
       Yes         0.195059
max    No          0.291990
       Yes         0.710345
dtype: float64

In [89]:
#the same thing but more short the previous
def f(group):
    return group.describe()

grouped.apply(f)

<ipython-input-89-28ed4c732e44>:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped.apply(f)


total_bill       tip  size   tip_pct
day  smoker                                            
Fri  No     count    4.000000  4.000000  4.00  4.000000
            mean    18.420000  2.812500  2.25  0.151650
            std      5.059282  0.898494  0.50  0.028123
            min     12.460000  1.500000  2.00  0.120385
            25%     15.100000  2.625000  2.00  0.137239
...                       ...       ...   ...       ...
Thur Yes    min     10.340000  2.000000  2.00  0.090014
            25%     13.510000  2.000000  2.00  0.148038
            50%     16.470000  2.560000  2.00  0.153846
            75%     19.810000  4.000000  2.00  0.194837
            max     43.110000  5.000000  4.00  0.241255

[64 rows x 4 columns]

##Suppresing the group keys

In [90]:
tips.groupby("smoker", group_keys=False).apply(top)

<ipython-input-90-de6df242f7c0>:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tips.groupby("smoker", group_keys=False).apply(top)


,total_bill,tip,smoker,day,time,size,tip_pct
232,11.61,3.39,No,Sat,Dinner,2,0.291990
149,7.51,2.00,No,Thur,Lunch,2,0.266312
51,10.29,2.60,No,Sun,Dinner,2,0.252672
185,20.69,5.00,No,Sun,Dinner,5,0.241663
88,24.71,5.85,No,Thur,Lunch,2,0.236746
172,7.25,5.15,Yes,Sun,Dinner,2,0.710345
178,9.60,4.00,Yes,Sun,Dinner,2,0.416667
67,3.07,1.00,Yes,Sat,Dinner,1,0.325733
183,23.17,6.50,Yes,Sun,Dinner,4,0.280535
109,14.31,4.00,Yes,Sat,Dinner,2,0.279525


In [91]:
tips.groupby("smoker").apply(top)

<ipython-input-91-2a8c000674ed>:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tips.groupby("smoker").apply(top)


total_bill   tip smoker   day    time  size   tip_pct
smoker                                                           
No     232       11.61  3.39     No   Sat  Dinner     2  0.291990
       149        7.51  2.00     No  Thur   Lunch     2  0.266312
       51        10.29  2.60     No   Sun  Dinner     2  0.252672
       185       20.69  5.00     No   Sun  Dinner     5  0.241663
       88        24.71  5.85     No  Thur   Lunch     2  0.236746
Yes    172        7.25  5.15    Yes   Sun  Dinner     2  0.710345
       178        9.60  4.00    Yes   Sun  Dinner     2  0.416667
       67         3.07  1.00    Yes   Sat  Dinner     1  0.325733
       183       23.17  6.50    Yes   Sun  Dinner     4  0.280535
       109       14.31  4.00    Yes   Sat  Dinner     2  0.279525

##Quantile and Bucket Analysis

In [92]:
frame = pd.DataFrame({"data1": np.random.standard_normal(1000),
                      "data2": np.random.standard_normal(1000)})
frame.head()

,data1,data2
0,-0.179470,-0.574325
1,0.097646,0.632000
2,-0.649847,1.224651
3,1.110452,0.099320
4,0.224790,-0.012620


In [99]:
#divide en 4 intervalos
quartiles = pd.cut(frame["data1"], 4)
quartiles.head(20)

,data1
0,"(-1.663, -0.136]"
1,"(-0.136, 1.392]"
2,"(-1.663, -0.136]"
3,"(-0.136, 1.392]"
4,"(-0.136, 1.392]"
5,"(-1.663, -0.136]"
6,"(-1.663, -0.136]"
7,"(-0.136, 1.392]"
8,"(-0.136, 1.392]"
9,"(-0.136, 1.392]"


In [100]:
def get_stats(group):
  return pd.DataFrame({"min": group.min(), "max": group.max(),
                       "count": group.count(), "mean": group.mean()})

grouped = frame.groupby(quartiles)
grouped.apply(get_stats)

<ipython-input-100-ed37e043a282>:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = frame.groupby(quartiles)


min       max  count      mean
data1                                                      
(-3.197, -1.663] data1 -3.191006 -1.675090     47 -2.054401
                 data2 -2.204628  1.637334     47 -0.236033
(-1.663, -0.136] data1 -1.642251 -0.140880    405 -0.744181
                 data2 -3.460643  2.913898    405 -0.010010
(-0.136, 1.392]  data1 -0.132380  1.381501    477  0.498011
                 data2 -3.307784  2.605533    477  0.020844
(1.392, 2.92]    data1  1.397545  2.919528     71  1.784529
                 data2 -2.121492  1.177008     71 -0.066594

In [101]:
#Same function
grouped.agg(["min", "max", "count", "mean"])

data1                               data2            \
                       min       max count      mean       min       max   
data1                                                                      
(-3.197, -1.663] -3.191006 -1.675090    47 -2.054401 -2.204628  1.637334   
(-1.663, -0.136] -1.642251 -0.140880   405 -0.744181 -3.460643  2.913898   
(-0.136, 1.392]  -0.132380  1.381501   477  0.498011 -3.307784  2.605533   
(1.392, 2.92]     1.397545  2.919528    71  1.784529 -2.121492  1.177008   

                                  
                 count      mean  
data1                             
(-3.197, -1.663]    47 -0.236033  
(-1.663, -0.136]   405 -0.010010  
(-0.136, 1.392]    477  0.020844  
(1.392, 2.92]       71 -0.066594

In [109]:
grouped.head()

,data1,data2
0,-0.179470,-0.574325
1,0.097646,0.632000
2,-0.649847,1.224651
3,1.110452,0.099320
4,0.224790,-0.012620
5,-0.211535,0.098511
6,-0.708417,-0.081991
7,1.137444,-0.395162
8,0.092479,-0.660369
12,-1.179984,0.151398


In [110]:
quartiles_samp = pd.qcut(frame["data1"], 4, labels=False)
quartiles_samp.head()


,data1
0,1
1,2
2,1
3,3
4,2


In [111]:
grouped = frame.groupby(quartiles_samp)
grouped.apply(get_stats)

min       max  count      mean
data1                                           
0     data1 -3.191006 -0.684576    250 -1.273310
      data2 -3.460643  2.813538    250 -0.031884
1     data1 -0.681589 -0.013893    250 -0.332082
      data2 -2.515035  2.913898    250  0.009512
2     data1 -0.011578  0.628867    250  0.281106
      data2 -2.928266  2.305655    250 -0.093203
3     data1  0.632506  2.919528    250  1.189496
      data2 -3.307784  2.605533    250  0.075843

###Filling Missing Values with Group- Specific Values
####fillna()

In [112]:
s = pd.Series(np.random.standard_normal(6))
s[::2] = np.nan
s

,0
0,NaN
1,-0.304039
2,NaN
3,0.356581
4,NaN
5,-0.115428


In [113]:
s.fillna(s.mean())

,0
0,-0.020962
1,-0.304039
2,-0.020962
3,0.356581
4,-0.020962
5,-0.115428


In [114]:
states = ["Ohio", "New York", "Vermont", "Florida",
          "Oregon", "Nevada", "California", "Idaho"]
group_key = ["East", "East", "East", "East",
             "West", "West", "West", "West"]
data = pd.Series(np.random.standard_normal(8), index=states)
data

,0
Ohio,0.797284
New York,0.077842
Vermont,-1.022765
Florida,0.461464
Oregon,0.067020
Nevada,-1.210810
California,-1.236304
Idaho,0.572445


In [115]:
data[["Vermont", "Nevada", "Idaho"]] = np.nan
data

,0
Ohio,0.797284
New York,0.077842
Vermont,NaN
Florida,0.461464
Oregon,0.067020
Nevada,NaN
California,-1.236304
Idaho,NaN


In [116]:
data.groupby(group_key).size()

,0
East,4
West,4


In [117]:
data.groupby(group_key).count()

,0
East,3
West,2


In [118]:
data.groupby(group_key).mean()

,0
East,0.445530
West,-0.584642


In [119]:
def fill_mean(group):
  return group.fillna(group.mean())
data.groupby(group_key).apply(fill_mean)

East  Ohio          0.797284
      New York      0.077842
      Vermont       0.445530
      Florida       0.461464
West  Oregon        0.067020
      Nevada       -0.584642
      California   -1.236304
      Idaho        -0.584642
dtype: float64

In [120]:
fill_values = {"East": 0.5, "West": -1}
def fill_func(group):
  return group.fillna(fill_values[group.name])
data.groupby(group_key).apply(fill_func)

East  Ohio          0.797284
      New York      0.077842
      Vermont       0.500000
      Florida       0.461464
West  Oregon        0.067020
      Nevada       -1.000000
      California   -1.236304
      Idaho        -1.000000
dtype: float64

###Random Sampling Permutation

In [131]:
suits = ["H", "S", "C", "D"]  # Hearts, Spades, Clubs, Diamonds
card_val = (list(range(1, 11)) + [10] * 3) * 4
base_names = ["A"] + list(range(2, 11)) + ["J", "K", "Q"]
cards = []
for suit in suits:
    cards.extend(str(num) + suit for num in base_names)
deck = pd.Series(card_val, index=cards)
deck.head(13)

,0
AH,1
2H,2
3H,3
4H,4
5H,5
6H,6
7H,7
8H,8
9H,9
10H,10


In [124]:
deck.count()

np.int64(52)

###.sample()
take and random elements

In [127]:
def draw(deck, n=5):
  return deck.sample(n)
draw(deck)

,0
5H,5
6C,6
3S,3
4D,4
4C,4


In [128]:
def get_suit(card):
 # last letter is suit
 return card[-1]
deck.groupby(get_suit).apply(draw, n=2)

C  3C      3
   8C      8
D  7D      7
   6D      6
H  3H      3
   10H    10
S  7S      7
   5S      5
dtype: int64

In [129]:
deck.groupby(get_suit, group_keys=False).apply(draw, n=2)

,0
AC,1
7C,7
QD,10
KD,10
AH,1
JH,10
2S,2
9S,9


###Group Weighted Average and Correlation

In [132]:
df = pd.DataFrame({"category": ["a", "a", "a", "a","b", "b", "b", "b"],
                   "data": np.random.standard_normal(8),
                   "weights": np.random.uniform(size=8)})
df

,category,data,weights
0,a,1.103948,0.889329
1,a,0.897205,0.848176
2,a,-0.976535,0.931303
3,a,1.014154,0.160853
4,b,-1.057106,0.137842
5,b,-0.331713,0.991747
6,b,-1.076208,0.786332
7,b,-0.364687,0.935775


In [133]:
grouped = df.groupby("category")
def get_wavg(group):
  return np.average(group["data"], weights=group["weights"])
grouped.apply(get_wavg)
#Cuando ciertos valores tienen más importancia o peso
#si los datos no tienen la misma relevancia.

<ipython-input-133-092023098d17>:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped.apply(get_wavg)


,0
category,
a,0.352142
b,-0.582885


##.info()
method is a convenient way to get an overview of the contents of a DataFrame.

In [135]:
close_px = pd.read_csv("examples/stock_px.csv", parse_dates=True, index_col=0)
close_px.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2214 entries, 2003-01-02 to 2011-10-14
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AAPL    2214 non-null   float64
 1   MSFT    2214 non-null   float64
 2   XOM     2214 non-null   float64
 3   SPX     2214 non-null   float64
dtypes: float64(4)
memory usage: 86.5 KB


In [136]:
close_px.tail(4)

,AAPL,MSFT,XOM,SPX
2011-10-11,400.29,27.00,76.27,1195.54
2011-10-12,402.19,26.96,77.16,1207.25
2011-10-13,408.43,27.18,76.37,1203.66
2011-10-14,422.00,27.27,78.11,1224.58


In [138]:
def spx_corr(group):
  return group.corrwith(group["SPX"])
rets = close_px.pct_change().dropna()
def get_year(x):
  return x.year
by_year = rets.groupby(get_year)
by_year.apply(spx_corr)

,AAPL,MSFT,XOM,SPX
2003,0.541124,0.745174,0.661265,1.0
2004,0.374283,0.588531,0.557742,1.0
2005,0.467540,0.562374,0.631010,1.0
2006,0.428267,0.406126,0.518514,1.0
2007,0.508118,0.658770,0.786264,1.0
2008,0.681434,0.804626,0.828303,1.0
2009,0.707103,0.654902,0.797921,1.0
2010,0.710105,0.730118,0.839057,1.0
2011,0.691931,0.800996,0.859975,1.0


**pct_change** we compute percent change on close_px using pct_change

In [140]:
def corr_aapl_msft(group):
  return group["AAPL"].corr(group["MSFT"])
by_year.apply(corr_aapl_msft)

,0
2003,0.480868
2004,0.259024
2005,0.300093
2006,0.161735
2007,0.417738
2008,0.611901
2009,0.432738
2010,0.571946
2011,0.581987


###Group-Wise Linear Regression
OLS which executes an ordinary least squares (OLS) regression on each chunk of data

In [142]:
import statsmodels.api as sm
def regress(data, yvar=None, xvars=None):
    Y = data[yvar]
    X = data[xvars]
    X["intercept"] = 1.
    result = sm.OLS(Y, X).fit()
    return result.params
by_year.apply(regress, yvar="AAPL", xvars=["SPX"])

,SPX,intercept
2003,1.195406,0.000710
2004,1.363463,0.004201
2005,1.766415,0.003246
2006,1.645496,0.000080
2007,1.198761,0.003438
2008,0.968016,-0.001110
2009,0.879103,0.002954
2010,1.052608,0.001261
2011,0.806605,0.001514


###Group Transforms and "Unwrapped" GroupBys

In [143]:
df = pd.DataFrame({'key': ['a', 'b', 'c'] * 4,'value': np.arange(12.)})
df

,key,value
0,a,0.0
1,b,1.0
2,c,2.0
3,a,3.0
4,b,4.0
5,c,5.0
6,a,6.0
7,b,7.0
8,c,8.0
9,a,9.0


In [144]:
g = df.groupby('key')['value']
g.mean()

,value
key,
a,4.5
b,5.5
c,6.5


###.transform()
transform works with functions that return Series, but the result must be the same size as the input, like apply()

In [145]:
def get_mean(group):
  return group.mean()
g.transform(get_mean)

,value
0,4.5
1,5.5
2,6.5
3,4.5
4,5.5
5,6.5
6,4.5
7,5.5
8,6.5
9,4.5


In [150]:
g.transform(get_mean)

,value
0,4.5
1,5.5
2,6.5
3,4.5
4,5.5
5,6.5
6,4.5
7,5.5
8,6.5
9,4.5


In [155]:
g.transform('mean')

,value
0,4.5
1,5.5
2,6.5
3,4.5
4,5.5
5,6.5
6,4.5
7,5.5
8,6.5
9,4.5


In [156]:
def times_two(group):
  return group * 2
g.transform(times_two)

,value
0,0.0
1,2.0
2,4.0
3,6.0
4,8.0
5,10.0
6,12.0
7,14.0
8,16.0
9,18.0


In [160]:
def get_ranks(group):
  return group.rank(ascending=False)
g.transform(get_ranks)

,value
0,4.0
1,4.0
2,4.0
3,3.0
4,3.0
5,3.0
6,2.0
7,2.0
8,2.0
9,1.0


In [162]:
def normalize(x):
  return (x - x.mean()) / x.std()
g.transform(normalize)

,value
0,-1.161895
1,-1.161895
2,-1.161895
3,-0.387298
4,-0.387298
5,-0.387298
6,0.387298
7,0.387298
8,0.387298
9,1.161895


In [163]:
g.apply(normalize)

key    
a    0    -1.161895
     3    -0.387298
     6     0.387298
     9     1.161895
b    1    -1.161895
     4    -0.387298
     7     0.387298
     10    1.161895
c    2    -1.161895
     5    -0.387298
     8     0.387298
     11    1.161895
Name: value, dtype: float64


.transform()	  Quieres mantener la forma original del DataFrame/Serie

---


.apply()	Quieres ver resultados agrupados jerárquicamente, o aplicar lógica más compleja

In [164]:
g.transform('mean')

,value
0,4.5
1,5.5
2,6.5
3,4.5
4,5.5
5,6.5
6,4.5
7,5.5
8,6.5
9,4.5


In [165]:
normalized = (df['value'] - g.transform('mean')) / g.transform('std')
normalized

,value
0,-1.161895
1,-1.161895
2,-1.161895
3,-0.387298
4,-0.387298
5,-0.387298
6,0.387298
7,0.387298
8,0.387298
9,1.161895


##Pivot Tables and Cross-Tabulation
**Pivot Table** It is a data summarization tool frequently found in spreadsheet programs and other data analysis software

In [166]:
tips.head()

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808


##.pivot_table()

In [167]:
tips.pivot_table(index=["day", "smoker"],
                 values=["size", "tip", "tip_pct", "total_bill"])

size       tip   tip_pct  total_bill
day  smoker                                          
Fri  No      2.250000  2.812500  0.151650   18.420000
     Yes     2.066667  2.714000  0.174783   16.813333
Sat  No      2.555556  3.102889  0.158048   19.661778
     Yes     2.476190  2.875476  0.147906   21.276667
Sun  No      2.929825  3.167895  0.160113   20.506667
     Yes     2.578947  3.516842  0.187250   24.120000
Thur No      2.488889  2.673778  0.160298   17.113111
     Yes     2.352941  3.030000  0.163863   19.190588

In [168]:
tips.pivot_table(index=["time", "day"], columns="smoker",
                 values=["tip_pct", "size"])

size             tip_pct          
smoker             No       Yes        No       Yes
time   day                                         
Dinner Fri   2.000000  2.222222  0.139622  0.165347
       Sat   2.555556  2.476190  0.158048  0.147906
       Sun   2.929825  2.578947  0.160113  0.187250
       Thur  2.000000       NaN  0.159744       NaN
Lunch  Fri   3.000000  1.833333  0.187735  0.188937
       Thur  2.500000  2.352941  0.160311  0.163863

**margins=True** This has the effect of adding All row and column labels, with corresponding values being the group statistics for all the data within a single tie

In [169]:
tips.pivot_table(index=["time", "day"], columns="smoker",
                 values=["tip_pct", "size"], margins=True)

size                       tip_pct                    
smoker             No       Yes       All        No       Yes       All
time   day                                                             
Dinner Fri   2.000000  2.222222  2.166667  0.139622  0.165347  0.158916
       Sat   2.555556  2.476190  2.517241  0.158048  0.147906  0.153152
       Sun   2.929825  2.578947  2.842105  0.160113  0.187250  0.166897
       Thur  2.000000       NaN  2.000000  0.159744       NaN  0.159744
Lunch  Fri   3.000000  1.833333  2.000000  0.187735  0.188937  0.188765
       Thur  2.500000  2.352941  2.459016  0.160311  0.163863  0.161301
All          2.668874  2.408602  2.569672  0.159328  0.163196  0.160803

aggfun=parameter
you can specified the function to do

In [170]:
tips.pivot_table(index=["time", "smoker"], columns="day",
                 values="tip_pct", aggfunc=len, margins=True)


day             Fri   Sat   Sun  Thur  All
time   smoker                             
Dinner No       3.0  45.0  57.0   1.0  106
       Yes      9.0  42.0  19.0   NaN   70
Lunch  No       1.0   NaN   NaN  44.0   45
       Yes      6.0   NaN   NaN  17.0   23
All            19.0  87.0  76.0  62.0  244

In [171]:
tips.pivot_table(index=["time", "size", "smoker"], columns="day",
                 values="tip_pct", fill_value=0)

day                      Fri       Sat       Sun      Thur
time   size smoker                                        
Dinner 1    No      0.000000  0.137931  0.000000  0.000000
            Yes     0.000000  0.325733  0.000000  0.000000
       2    No      0.139622  0.162705  0.168859  0.159744
            Yes     0.171297  0.148668  0.207893  0.000000
       3    No      0.000000  0.154661  0.152663  0.000000
            Yes     0.000000  0.144995  0.152660  0.000000
       4    No      0.000000  0.150096  0.148143  0.000000
            Yes     0.117750  0.124515  0.193370  0.000000
       5    No      0.000000  0.000000  0.206928  0.000000
            Yes     0.000000  0.106572  0.065660  0.000000
       6    No      0.000000  0.000000  0.103799  0.000000
Lunch  1    No      0.000000  0.000000  0.000000  0.181728
            Yes     0.223776  0.000000  0.000000  0.000000
       2    No      0.000000  0.000000  0.000000  0.166005
            Yes     0.181969  0.000000  0.000000  0.158843
       3    No      0.187735  0.000000  0.000000  0.084246
            Yes     0.000000  0.000000  0.000000  0.204952
       4    No      0.000000  0.000000  0.000000  0.138919
            Yes     0.000000  0.000000  0.000000  0.155410
       5    No      0.000000  0.000000  0.000000  0.121389
       6    No      0.000000  0.000000  0.000000  0.173706

#Cross-Tabulation (Crosstab)
**cross-tabulation** (or crosstab for short) is a special case of a pivot table that computes group frequencies

In [173]:
from io import StringIO
data = """Sample  Nationality  Handedness
          1   USA  Right-handed
          2   Japan    Left-handed
          3   USA  Right-handed
          4   Japan    Right-handed
          5   Japan    Left-handed
          6   Japan    Right-handed
          7   USA  Right-handed
          8   USA  Left-handed
          9   Japan    Right-handed
          10  USA  Right-handed"""
data = pd.read_table(StringIO(data), sep="\s+")
data

,Sample,Nationality,Handedness
0,1,USA,Right-handed
1,2,Japan,Left-handed
2,3,USA,Right-handed
3,4,Japan,Right-handed
4,5,Japan,Left-handed
5,6,Japan,Right-handed
6,7,USA,Right-handed
7,8,USA,Left-handed
8,9,Japan,Right-handed
9,10,USA,Right-handed


##pandas.crosstab()

In [174]:
pd.crosstab(data["Nationality"], data["Handedness"], margins=True)

Handedness,Left-handed,Right-handed,All
Nationality,,,
Japan,2,3,5
USA,1,4,5
All,3,7,10


In [175]:
pd.crosstab([tips["time"], tips["day"]], tips["smoker"], margins=True)

smoker        No  Yes  All
time   day                
Dinner Fri     3    9   12
       Sat    45   42   87
       Sun    57   19   76
       Thur    1    0    1
Lunch  Fri     1    6    7
       Thur   44   17   61
All          151   93  244